# Data Ingestion, Cleaning & Preprocessing with Pandas

**Dataset:** Retail Sales transactions (10,000+ rows)

**Steps:** load raw data → profile (BEFORE) → fix types → handle missing values → handle outliers → feature engineering → profile (AFTER) → export `clean_dataset.csv`

> Agar `raw_retail_orders.csv` file isi folder me hai to wahi load hoti hai. Nahi to demo messy dataset generate hota hai taaki notebook end-to-end chal sake.

In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 20)

RAW_PATH = "raw_retail_orders.csv"
CLEAN_PATH = "clean_dataset.csv"

## 1. Load raw dataset

In [3]:
def make_messy_dataset(n=12000, seed=7):
    """Realistic messy retail data: nulls, outliers, wrong types, duplicates, inconsistent text."""
    rng = np.random.default_rng(seed)
    cats = ["Electronics", "Clothing", "Home & Kitchen", "Beauty", "Sports", "Grocery"]
    base_price = {"Electronics": 9000, "Clothing": 1500, "Home & Kitchen": 2500, "Beauty": 800, "Sports": 2000, "Grocery": 300}
    category = rng.choice(cats, n)
    price = np.array([base_price[c] for c in category]) * rng.uniform(0.6, 1.4, n)
    cost = price * rng.uniform(0.5, 0.85, n)
    qty = rng.integers(1, 6, n).astype(float)
    dates = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 700, n), unit="D")
    df = pd.DataFrame({
        "Order ID": np.arange(100001, 100001 + n), "Customer ID": rng.integers(1000, 4000, n).astype(float),
        "Order Date": dates, "Category": category,
        "Region": rng.choice(["North", "South", "East", "West"], n),
        "Quantity": qty, "Unit Price": price.round(2), "Cost Price": cost.round(2),
        "Order Status": rng.choice(["Delivered", "Shipped", "Cancelled", "Returned"], n, p=[.7, .15, .1, .05]),
    })
    # --- inject mess ---
    fmt = rng.choice(["iso", "dmy", "long"], n, p=[.5, .3, .2])
    d = df["Order Date"]
    df["Order Date"] = np.where(fmt == "iso", d.dt.strftime("%Y-%m-%d"),
                         np.where(fmt == "dmy", d.dt.strftime("%d/%m/%Y"), d.dt.strftime("%b %d, %Y")))
    df.loc[rng.choice(n, 120, replace=False), "Order Date"] = "N/A"
    df["Unit Price"] = df["Unit Price"].astype(object)
    s = rng.choice(n, 1500, replace=False)
    df.loc[s, "Unit Price"] = ["$" + f"{float(v):,.2f}" for v in df.loc[s, "Unit Price"]]
    for col, k in [("Unit Price", 350), ("Cost Price", 400), ("Quantity", 250), ("Customer ID", 200), ("Category", 220), ("Region", 180)]:
        df.loc[rng.choice(n, k, replace=False), col] = np.nan
    df.loc[rng.choice(n, 60, replace=False), "Quantity"] = 999
    df.loc[rng.choice(n, 25, replace=False), "Quantity"] = -3
    o = rng.choice(n, 50, replace=False)
    df.loc[o, "Unit Price"] = pd.to_numeric(df.loc[o, "Unit Price"], errors="coerce") * 40
    df["Category"] = df["Category"].apply(lambda v: v if pd.isna(v) else rng.choice([v, v.lower(), v.upper(), " " + v + " "], p=[.7, .1, .1, .1]))
    df["Order Status"] = df["Order Status"].apply(lambda v: rng.choice([v, v.lower(), v.upper() + " "], p=[.8, .1, .1]))
    df = pd.concat([df, df.sample(200, random_state=1)], ignore_index=True)   # duplicates
    return df.sample(frac=1, random_state=3).reset_index(drop=True)

if Path(RAW_PATH).exists():
    raw = pd.read_csv(RAW_PATH)
    print("Loaded real file:", RAW_PATH)
else:
    raw = make_messy_dataset()
    print("File nahi mili, demo messy dataset generate hua. Asli file ho to RAW_PATH me daalo.")
print("Shape:", raw.shape)
print(raw.head(8).to_string())

File nahi mili, demo messy dataset generate hua. Asli file ho to RAW_PATH me daalo.
Shape: (12200, 9)
   Order ID  Customer ID    Order Date          Category Region  Quantity Unit Price  Cost Price Order Status
0    100314       3184.0  Feb 21, 2025          Clothing   East       1.0     931.15      507.09    Delivered
1    111778       2670.0           N/A    Home & Kitchen   East       2.0  $2,670.54     2060.62    Delivered
2    103866          NaN    04/08/2025         Clothing    East       4.0        NaN     1314.39      Shipped
3    105545       3151.0    2024-04-07           Grocery  South       1.0    $394.52      282.61    Delivered
4    108199          NaN    2025-05-26   Home & Kitchen   North       1.0    2097.48     1752.22    Delivered
5    101110       3317.0    09/01/2024          Clothing   West       3.0    1375.74      835.13    Delivered
6    105017       3565.0    2024-03-24          Clothing   East       5.0    2015.25     1010.43   DELIVERED 
7    111484       

## 2. BEFORE cleaning: profile

In [5]:
def profile(d, title):
    print(f"===== {title} =====")
    print("Rows x Cols:", d.shape)
    print("Duplicate rows:", int(d.duplicated().sum()))
    p = pd.DataFrame({"dtype": d.dtypes.astype(str), "nulls": d.isna().sum(), "null_%": (d.isna().mean() * 100).round(2)})
    print(p.to_string())

profile(raw, "BEFORE")
before = raw.copy()
before_summary = {"rows": len(raw), "duplicates": int(raw.duplicated().sum()), "total_nulls": int(raw.isna().sum().sum())}

===== BEFORE =====
Rows x Cols: (12200, 9)
Duplicate rows: 200
                dtype  nulls  null_%
Order ID        int64      0    0.00
Customer ID   float64    203    1.66
Order Date        str      0    0.00
Category          str    225    1.84
Region            str    184    1.51
Quantity      float64    256    2.10
Unit Price     object    359    2.94
Cost Price    float64    405    3.32
Order Status      str      0    0.00


## 3. Cleaning

### 3.1 Standardize column names and text

In [8]:
df = raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")

# text columns: trim spaces + consistent case
for c in ["category", "region", "order_status"]:
    df[c] = df[c].astype("string").str.strip().str.title()
print(df["category"].value_counts(dropna=False).to_string())
print(df["order_status"].value_counts(dropna=False).to_string())

category
Clothing          2073
Grocery           2019
Home & Kitchen    2005
Sports            1999
Electronics       1967
Beauty            1912
<NA>               225
order_status
Delivered    8589
Shipped      1800
Cancelled    1219
Returned      592


### 3.2 Fix incorrect data types

In [10]:
# Prices like "$1,250.00" -> float
for c in ["unit_price", "cost_price", "quantity"]:
    df[c] = pd.to_numeric(df[c].astype("string").str.replace(r"[$,\s]", "", regex=True), errors="coerce")

# Mixed date formats -> datetime ("N/A" etc. become NaT)
df["order_date"] = pd.to_datetime(df["order_date"], format="mixed", dayfirst=True, errors="coerce")
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
df["order_id"] = df["order_id"].astype("Int64")
print(df.dtypes.to_string())

order_id                 Int64
customer_id              Int64
order_date      datetime64[us]
category                string
region                  string
quantity               Float64
unit_price             Float64
cost_price             Float64
order_status            string


### 3.3 Remove duplicates

In [12]:
n0 = len(df)
df = df.drop_duplicates().drop_duplicates(subset="order_id", keep="first").reset_index(drop=True)
print(f"Duplicates removed: {n0 - len(df)}  |  rows now: {len(df)}")

Duplicates removed: 200  |  rows now: 12000


### 3.4 Missing values
Rules: `order_date` missing → row drop (time analysis ke liye zaroori). Numeric → category-wise median. Categorical → `Unknown`. `customer_id` missing → `<NA>` rakha (guess karna galat hoga).

In [14]:
# invalid quantities (<=0) and prices (<=0) are errors, treat as missing
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["unit_price"] <= 0, "unit_price"] = np.nan
df.loc[df["cost_price"] <= 0, "cost_price"] = np.nan

# 1) categorical first (so category-wise median works)
df["category"] = df["category"].fillna("Unknown")
df["region"] = df["region"].fillna("Unknown")

# 2) drop rows without a usable date
dropped = int(df["order_date"].isna().sum())
df = df.dropna(subset=["order_date"]).reset_index(drop=True)
print("Rows dropped (no valid order_date):", dropped)

# 3) numeric: median by category
for c in ["unit_price", "cost_price", "quantity"]:
    df[c] = df[c].fillna(df.groupby("category")[c].transform("median"))
    df[c] = df[c].fillna(df[c].median())
print("Remaining nulls:\n", df.isna().sum().to_string())

Rows dropped (no valid order_date): 120
Remaining nulls:
 order_id          0
customer_id     199
order_date        0
category          0
region            0
quantity          0
unit_price        0
cost_price        0
order_status      0


### 3.5 Outliers (IQR capping per category)

In [16]:
def cap_iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return s.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

out_report = {}
for c in ["quantity", "unit_price", "cost_price"]:
    capped = df.groupby("category")[c].transform(cap_iqr)
    out_report[c] = int((capped != df[c]).sum())
    df[c] = capped
df["quantity"] = df["quantity"].round().astype(int)
print("Values capped per column:", out_report)
# cost can never exceed price
df["cost_price"] = df[["cost_price", "unit_price"]].min(axis=1)

Values capped per column: {'quantity': 57, 'unit_price': 75, 'cost_price': 29}


## 4. Feature engineering

In [18]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.month_name()
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)

df["revenue"] = (df["quantity"] * df["unit_price"]).round(2)
df["total_cost"] = (df["quantity"] * df["cost_price"]).round(2)
df["profit"] = (df["revenue"] - df["total_cost"]).round(2)
df["profit_margin_pct"] = ((df["profit"] / df["revenue"]) * 100).round(2)
print(df[["order_date", "order_year", "order_month", "quantity", "unit_price", "revenue", "profit", "profit_margin_pct"]].head(8).to_string())

  order_date  order_year  order_month  quantity  unit_price   revenue   profit  profit_margin_pct
0 2025-02-21        2025            2         1      931.15    931.15   424.06              45.54
1 2025-08-04        2025            8         4     1496.91   5987.64   730.08              12.19
2 2024-07-04        2024            7         1      394.52    394.52   111.91              28.37
3 2025-05-26        2025            5         1     2097.48   2097.48   345.26              16.46
4 2024-01-09        2024            1         3     1375.74   4127.22  1621.83               39.3
5 2024-03-24        2024            3         5     2015.25  10076.25   5024.1              49.86
6 2024-02-02        2024            2         5      242.99   1214.95   204.85              16.86
7 2024-12-30        2024           12         3     3210.51   9631.53  2242.56              23.28


## 5. AFTER cleaning: profile

In [20]:
profile(df, "AFTER")
print()
print(df[["quantity", "unit_price", "cost_price", "revenue", "profit", "profit_margin_pct"]].describe().round(2).to_string())

===== AFTER =====
Rows x Cols: (11880, 17)
Duplicate rows: 0
                            dtype  nulls  null_%
order_id                    Int64      0    0.00
customer_id                 Int64    199    1.68
order_date         datetime64[us]      0    0.00
category                   string      0    0.00
region                     string      0    0.00
quantity                    int64      0    0.00
unit_price                Float64      0    0.00
cost_price                Float64      0    0.00
order_status               string      0    0.00
order_year                  int32      0    0.00
order_month                 int32      0    0.00
month_name                    str      0    0.00
year_month                    str      0    0.00
revenue                   Float64      0    0.00
total_cost                Float64      0    0.00
profit                    Float64      0    0.00
profit_margin_pct         Float64      0    0.00

       quantity  unit_price  cost_price   revenue    pro

## 6. Before vs After

In [22]:
cmp = pd.DataFrame({
    "BEFORE": [len(before), before_summary["duplicates"], before_summary["total_nulls"],
               before["Unit Price"].dtype.name, before["Order Date"].dtype.name, before["Quantity"].max()],
    "AFTER":  [len(df), int(df.duplicated().sum()), int(df.drop(columns="customer_id").isna().sum().sum()),
               df["unit_price"].dtype.name, df["order_date"].dtype.name, df["quantity"].max()],
}, index=["rows", "duplicate rows", "null cells (excl. customer_id)", "unit_price dtype", "order_date dtype", "max quantity"])
print(cmp.to_string())

                                BEFORE           AFTER
rows                             12200           11880
duplicate rows                     200               0
null cells (excl. customer_id)    1632               0
unit_price dtype                object         Float64
order_date dtype                   str  datetime64[us]
max quantity                     999.0               7


## 7. Export clean dataset

In [24]:
df.to_csv(CLEAN_PATH, index=False)
check = pd.read_csv(CLEAN_PATH, parse_dates=["order_date"])
print(f"Saved {CLEAN_PATH}: {check.shape[0]} rows x {check.shape[1]} columns")
print(list(check.columns))

Saved clean_dataset.csv: 11880 rows x 17 columns
['order_id', 'customer_id', 'order_date', 'category', 'region', 'quantity', 'unit_price', 'cost_price', 'order_status', 'order_year', 'order_month', 'month_name', 'year_month', 'revenue', 'total_cost', 'profit', 'profit_margin_pct']
